# Intelligent Course Recommendation System (RAG)
This notebook builds a course recommendation workflow with Retrieval-Augmented Generation (RAG), structured filtering, and an interactive search interface.

**What this notebook now includes:**
1. Data cleaning utilities for meeting days, time ranges, and review aggregation.
2. A `CourseRAG` pipeline that combines vector retrieval, cross-encoder reranking, and practical scoring signals.
3. Optional Groq-based JSON recommendation summaries.
4. A widget-based interface for exploring results by topic, schedule, rating, workload, and review coverage.

**Notebook flow:**
1. Setup and file paths.
2. Data cleaning helpers.
3. Review aggregation and course loading.
4. Retrieval and ranking helpers.
5. `CourseRAG` indexing, search, and optional LLM advice.
6. Interactive UI construction and rendering.


## 0. Setup and Configuration
Load dependencies, environment variables, and dataset paths used throughout the notebook.


In [1]:
import os
import csv
import json
import math
from html import escape
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, CrossEncoder
import ipywidgets as widgets
from IPython.display import display
from dotenv import load_dotenv

# Load environment variables so optional API-based features can be enabled.
load_dotenv()

# Enable Groq only when the API key is available in the environment.
USE_GROQ = bool(os.environ.get("GROQ_API_KEY"))
if USE_GROQ:
    from groq import Groq

# Keep dataset and vector database paths in one place for easier maintenance.
DATA_DIR = Path(".")
COURSES_CSV = DATA_DIR / "courses_full_dataset_combined_courses.csv"
REVIEWS_CSV = DATA_DIR / "course_review.csv"
DB_PATH = DATA_DIR / "chroma_db"

# Fall back to the alternate dataset name when the primary file is unavailable.
if not COURSES_CSV.exists():
    COURSES_CSV = DATA_DIR / "cmu_labeled_llm_final.csv"

print(f"Courses CSV path: {COURSES_CSV} - Exists: {COURSES_CSV.exists()}")
print(f"Reviews CSV path: {REVIEWS_CSV} - Exists: {REVIEWS_CSV.exists()}")


Courses CSV path: courses_full_dataset_combined_courses.csv - Exists: True
Reviews CSV path: course_review.csv - Exists: True


## 1. Data Cleaning Utilities
Normalize schedule-related fields so downstream filtering and ranking can work with consistent values.


In [2]:
def parse_days(weekday_str: str) -> List[str]:
    """Extract valid weekday codes such as M, T, W, R, and F."""
    if not weekday_str or weekday_str == "TBA":
        return []

    valid_chars = {"M", "T", "W", "R", "F", "S", "U"}
    return [char for char in str(weekday_str).upper() if char in valid_chars]


def normalize_time_str(value: str) -> str:
    """Standardize time strings into a predictable 'H:MM AM/PM' format."""
    if value is None:
        return ""

    clean = str(value).encode("ascii", "ignore").decode("ascii").strip().upper()
    clean = " ".join(clean.replace(".", "").split())
    if clean in {"", "TBA", "N/A", "NA"}:
        return ""

    parts = clean.split()
    if len(parts) != 2:
        return clean

    time_part, period = parts
    if ":" not in time_part:
        time_part = f"{time_part}:00"
    return f"{time_part} {period}"


def parse_time_slots(start_str: str, end_str: str) -> List[str]:
    """Expand a class meeting window into 30-minute time slots."""
    start_str = normalize_time_str(start_str)
    end_str = normalize_time_str(end_str)
    if not start_str or not end_str:
        return []

    def to_minutes(time_str: str) -> int:
        try:
            time_part, period = time_str.split()
            hour, minute = map(int, time_part.split(":"))
            if period == "PM" and hour != 12:
                hour += 12
            if period == "AM" and hour == 12:
                hour = 0
            return hour * 60 + minute
        except Exception:
            return -1

    start_minutes = to_minutes(start_str)
    end_minutes = to_minutes(end_str)
    if start_minutes == -1 or end_minutes == -1 or start_minutes >= end_minutes:
        return []

    slots = []
    current_minutes = start_minutes
    while current_minutes < end_minutes:
        hour, minute = divmod(current_minutes, 60)
        period = "PM" if hour >= 12 else "AM"
        display_hour = hour - 12 if hour > 12 else (12 if hour == 0 else hour)
        slots.append(f"{display_hour}:{minute:02d} {period}")
        current_minutes += 30
    return slots


## 2. Review Aggregation and Course Loading
Merge review statistics into the course records so retrieval can use both content and student signal features.


In [3]:
def aggregate_reviews(reviews_path: Path) -> Dict[str, Dict[str, Any]]:
    """Group raw review rows into course-level summary statistics."""
    if not reviews_path.exists():
        return {}

    reviews_df = pd.read_csv(reviews_path, encoding="utf-8", on_bad_lines="skip")
    if reviews_df.empty or "CourseID" not in reviews_df.columns:
        return {}

    numeric_columns = ["OverallRating", "WorkloadHours", "UtilityRating", "InterestRating"]
    for column in numeric_columns:
        if column in reviews_df.columns:
            reviews_df[column] = pd.to_numeric(reviews_df[column], errors="coerce")

    grouped_reviews = {}
    for course_id, group in reviews_df.groupby("CourseID"):
        comments = []
        if "Comment" in group.columns:
            comments = [
                str(comment).strip().replace("\n", " ")
                for comment in group["Comment"].dropna().tolist()
                if str(comment).strip()
            ]

        grouped_reviews[str(course_id)] = {
            "review_count": int(len(group)),
            "avg_rating": round(group["OverallRating"].dropna().mean(), 2)
            if "OverallRating" in group.columns and not group["OverallRating"].dropna().empty
            else None,
            "avg_workload_hours": round(group["WorkloadHours"].dropna().mean(), 2)
            if "WorkloadHours" in group.columns and not group["WorkloadHours"].dropna().empty
            else None,
            "avg_utility": round(group["UtilityRating"].dropna().mean(), 2)
            if "UtilityRating" in group.columns and not group["UtilityRating"].dropna().empty
            else None,
            "avg_interest": round(group["InterestRating"].dropna().mean(), 2)
            if "InterestRating" in group.columns and not group["InterestRating"].dropna().empty
            else None,
            "review_snippets": comments[:3],
        }
    return grouped_reviews


def load_and_clean_data(courses_path: Path, reviews_path: Path) -> List[Dict[str, Any]]:
    """Load course rows and attach normalized schedule plus review metadata."""
    courses = []
    review_lookup = aggregate_reviews(reviews_path)

    with open(courses_path, "r", encoding="utf-8", errors="replace") as file_obj:
        reader = csv.DictReader(file_obj)
        for row in reader:
            weekday = row.get("weekday", "")
            start = row.get("start", "")
            end = row.get("end", "")

            # Store parsed schedule features for filtering and matching.
            row["_days"] = parse_days(weekday)
            row["_times"] = parse_time_slots(start, end)

            clean_start = normalize_time_str(start)
            clean_end = normalize_time_str(end)
            row["_meetingTime"] = f"{weekday} {clean_start}-{clean_end}" if weekday and clean_start else "TBA"

            # Convert the serialized skills field into a real list.
            skills_str = str(row.get("skills", "")).replace("[", "").replace("]", "").replace("'", "")
            row["skills"] = [skill.strip() for skill in skills_str.split(",") if skill.strip()]

            # Merge review-derived features into each course record.
            review_summary = review_lookup.get(str(row.get("course_id", "")).strip(), {})
            row["review_count"] = review_summary.get("review_count", 0)
            row["avg_rating"] = review_summary.get("avg_rating")
            row["avg_workload_hours"] = review_summary.get("avg_workload_hours")
            row["avg_utility"] = review_summary.get("avg_utility")
            row["avg_interest"] = review_summary.get("avg_interest")
            row["review_snippets"] = review_summary.get("review_snippets", [])
            courses.append(row)

    print(f"Loaded {len(courses)} courses.")
    print(f"Aggregated reviews for {len(review_lookup)} unique courses.")
    return courses


COURSES = load_and_clean_data(COURSES_CSV, REVIEWS_CSV)
COURSES_BY_ID = {
    str(course["course_id"]): course
    for course in COURSES
    if str(course.get("course_id", "")).strip()
}


Loaded 8450 courses.
Aggregated reviews for 3145 unique courses.


## 3. Retrieval and Ranking Helpers
Keep scoring, filtering, and explanation logic outside the main class so the retrieval pipeline is easier to read and maintain.


In [4]:
def create_course_document(course: Dict[str, Any]) -> str:
    """Build the text representation that will be embedded and indexed."""
    parts = []
    if course.get("course_id"):
        parts.append(f"Course: {course['course_id']}")
    if course.get("course_name"):
        parts.append(f"Title: {course['course_name']}")

    description = course.get("description_clean") or course.get("description", "")
    if description:
        parts.append(f"Description: {description}")
    if course.get("industry"):
        parts.append(f"Industry: {course['industry']}")
    if course.get("level"):
        parts.append(f"Level: {course['level']}")
    if course.get("skills"):
        parts.append(f"Skills: {', '.join(course['skills'])}")
    if course.get("prerequisites"):
        parts.append(f"Prerequisites: {course['prerequisites']}")
    if course.get("_meetingTime") and course.get("_meetingTime") != "TBA":
        parts.append(f"Meeting time: {course['_meetingTime']}")

    review_count = course.get("review_count", 0)
    if review_count:
        parts.append(f"Review count: {review_count}")
    if course.get("avg_rating") is not None:
        parts.append(f"Average rating: {course['avg_rating']}/5")
    if course.get("avg_workload_hours") is not None:
        parts.append(f"Average workload hours: {course['avg_workload_hours']} per week")
    if course.get("avg_utility") is not None:
        parts.append(f"Average utility rating: {course['avg_utility']}/5")
    if course.get("avg_interest") is not None:
        parts.append(f"Average interest rating: {course['avg_interest']}/5")
    if course.get("review_snippets"):
        parts.append(f"Student feedback: {' || '.join(course['review_snippets'])}")

    return " | ".join(parts)


def matches_preferences(course: Dict[str, Any], preference_filter: Dict[str, float]) -> bool:
    """Apply hard preference constraints before expensive reranking."""
    if not preference_filter:
        return True

    min_rating = preference_filter.get("min_rating", 0)
    max_workload = preference_filter.get("max_workload")
    min_reviews = preference_filter.get("min_reviews", 0)

    avg_rating = course.get("avg_rating")
    workload = course.get("avg_workload_hours")
    review_count = course.get("review_count", 0)

    if avg_rating is not None and avg_rating < min_rating:
        return False
    if max_workload is not None and workload is not None and workload > max_workload:
        return False
    if review_count < min_reviews:
        return False
    return True


def tokenize_text(text: str) -> List[str]:
    """Tokenize short text snippets for lightweight keyword overlap scoring."""
    clean = str(text or "").lower()
    for char in ",./;:!?()[]{}|_-":
        clean = clean.replace(char, " ")
    return [token for token in clean.split() if len(token) > 2]


def sigmoid(value: float) -> float:
    return 1 / (1 + math.exp(-value))


def normalize_cross_score(score: float) -> float:
    """Map raw cross-encoder output into a stable 0-1 range."""
    return sigmoid(score / 2.2)


def score_keyword_overlap(query: str, course: Dict[str, Any]) -> float:
    """Measure how much of the query vocabulary appears in the course text."""
    query_tokens = set(tokenize_text(query))
    if not query_tokens:
        return 0.0

    text_parts = [
        course.get("course_name", ""),
        course.get("industry", ""),
        course.get("level", ""),
        " ".join(course.get("skills", [])),
        course.get("description_clean") or course.get("description", ""),
    ]
    course_tokens = set(tokenize_text(" ".join(text_parts)))
    if not course_tokens:
        return 0.0
    return len(query_tokens & course_tokens) / len(query_tokens)


def analyze_schedule_fit(course: Dict[str, Any], schedule_filter: Dict[str, Any]) -> Dict[str, Any]:
    """Compute whether a course is compatible with the selected schedule."""
    if not schedule_filter:
        return {
            "eligible": True,
            "score": 1.0,
            "day_score": None,
            "time_score": None,
            "matched_days": [],
            "selected_days": [],
            "has_time_filter": False,
            "summary": "No schedule filter applied.",
        }

    selected_days = set(schedule_filter.get("selected_days") or [])
    selected_slots = set(schedule_filter.get("time_slots") or [])
    has_day_filter = bool(selected_days)
    has_time_filter = bool(selected_slots)

    course_days = set(course.get("_days") or [])
    course_slots = set(course.get("_times") or [])

    matched_days = sorted(course_days & selected_days)
    day_score = None
    time_score = None
    eligible = True

    if has_day_filter:
        day_score = len(matched_days) / max(len(selected_days), 1)
        eligible = eligible and day_score > 0

    if has_time_filter:
        if course_slots:
            overlap = course_slots & selected_slots
            time_score = len(overlap) / max(len(course_slots), 1)
        else:
            time_score = 0.0
        eligible = eligible and time_score > 0

    active_scores = [score for score in [day_score, time_score] if score is not None]
    score = sum(active_scores) / len(active_scores) if active_scores else 1.0

    if has_day_filter and has_time_filter:
        summary = (
            f"Fits {len(matched_days)}/{len(selected_days)} selected days and "
            f"{int(round((time_score or 0) * 100))}% of the class meeting window."
        )
    elif has_day_filter:
        summary = f"Fits {len(matched_days)}/{len(selected_days)} selected days."
    else:
        summary = f"Fits {int(round((time_score or 0) * 100))}% of the class meeting window."

    return {
        "eligible": eligible,
        "score": score,
        "day_score": day_score,
        "time_score": time_score,
        "matched_days": matched_days,
        "selected_days": sorted(selected_days),
        "has_time_filter": has_time_filter,
        "summary": summary,
    }


def score_rating(course: Dict[str, Any]) -> float:
    rating = course.get("avg_rating")
    return 0.55 if rating is None else max(0.0, min(rating / 5, 1.0))


def score_utility(course: Dict[str, Any]) -> float:
    utility = course.get("avg_utility")
    return 0.5 if utility is None else max(0.0, min(utility / 5, 1.0))


def score_reviews(course: Dict[str, Any]) -> float:
    review_count = course.get("review_count", 0)
    return max(0.0, min(review_count / 20, 1.0))


def score_workload(course: Dict[str, Any], preference_filter: Dict[str, float] = None) -> float:
    """Reward lighter workloads, especially when the user set a cap."""
    workload = course.get("avg_workload_hours")
    if workload is None:
        return 0.45

    preferred_cap = None if not preference_filter else preference_filter.get("max_workload")
    if preferred_cap is not None:
        if workload <= preferred_cap:
            return 1.0 - min(workload / max(preferred_cap, 1), 1.0) * 0.35
        overflow = workload - preferred_cap
        return max(0.0, 0.55 - overflow / max(preferred_cap, 1))
    return max(0.0, 1 - min(workload, 20) / 20)


def blend_scores(
    semantic_score: float,
    keyword_score: float,
    schedule_score: float,
    rating_score: float,
    utility_score: float,
    workload_score: float,
    review_score: float,
) -> Dict[str, float]:
    """Combine semantic relevance with practical course signals."""
    component_scores = {
        "semantic": semantic_score,
        "keyword": keyword_score,
        "schedule": schedule_score,
        "rating": rating_score,
        "utility": utility_score,
        "workload": workload_score,
        "reviews": review_score,
    }
    weights = {
        "semantic": 0.40,
        "keyword": 0.16,
        "schedule": 0.16,
        "rating": 0.12,
        "utility": 0.07,
        "workload": 0.05,
        "reviews": 0.04,
    }
    final_score = sum(component_scores[name] * weights[name] for name in weights)
    return {"final": final_score, **component_scores}


def build_reason_tags(course: Dict[str, Any], scores: Dict[str, float], schedule_info: Dict[str, Any]) -> List[str]:
    """Generate short human-readable reasons for why a course ranked well."""
    tags = []
    if scores["semantic"] >= 0.8:
        tags.append("Strong topic match")
    elif scores["keyword"] >= 0.35:
        tags.append("Clear skill overlap")

    if schedule_info and schedule_info.get("score", 1) >= 0.99 and schedule_info.get("selected_days"):
        tags.append("Fits your selected schedule")
    elif schedule_info and schedule_info.get("score", 1) >= 0.6 and (
        schedule_info.get("selected_days") or schedule_info.get("has_time_filter")
    ):
        tags.append("Partially fits your schedule")

    if (course.get("avg_rating") or 0) >= 4.4:
        tags.append("Highly rated by students")
    if (course.get("avg_utility") or 0) >= 4.2:
        tags.append("High utility signal")
    if course.get("avg_workload_hours") is not None and course.get("avg_workload_hours") <= 8:
        tags.append("Light weekly workload")
    if course.get("review_count", 0) >= 5:
        tags.append("Backed by multiple reviews")
    return tags[:3] or ["Balanced overall fit"]


## 4. The CourseRAG Engine
Load models, build the vector index, retrieve candidates, rerank them, and optionally generate a structured LLM summary.


In [5]:
class CourseRAG:
    def __init__(self, db_path: str, use_groq: bool = False):
        self.db_path = Path(db_path)
        self.db_path.mkdir(exist_ok=True)

        self.client = chromadb.PersistentClient(
            path=str(self.db_path),
            settings=Settings(anonymized_telemetry=False, allow_reset=True),
        )
        self.collection = self.client.get_or_create_collection(
            name="courses_v4",
            metadata={"hnsw:space": "cosine"},
        )

        print("Loading Bi-Encoder...")
        self.bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        print("Loading Cross-Encoder for re-ranking...")
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

        self.use_groq = use_groq
        if self.use_groq:
            self.llm_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
            self.llm_model = "llama-3.3-70b-versatile"

    def ingest_data(self, courses: List[Dict[str, Any]]) -> None:
        """Embed all unique courses and write them into ChromaDB."""
        if self.collection.count() > 0:
            print(f"Collection already populated with {self.collection.count()} items. Skipping ingest.")
            return

        documents, metadatas, ids = [], [], []
        seen_ids = set()

        for index, row in enumerate(courses):
            course_id = str(row.get("course_id", f"idx_{index}"))
            if course_id in seen_ids:
                continue

            seen_ids.add(course_id)
            documents.append(create_course_document(row))
            ids.append(course_id)
            metadatas.append({
                "course_name": row.get("course_name", ""),
                "industry": row.get("industry", ""),
                "level": row.get("level", ""),
            })

        print(f"Embedding {len(documents)} unique documents...")
        embeddings = self.bi_encoder.encode(documents, show_progress_bar=True).tolist()

        batch_size = 5000
        for batch_idx in range(math.ceil(len(ids) / batch_size)):
            start = batch_idx * batch_size
            end = (batch_idx + 1) * batch_size
            self.collection.upsert(
                ids=ids[start:end],
                embeddings=embeddings[start:end],
                documents=documents[start:end],
                metadatas=metadatas[start:end],
            )
        print("Ingestion complete.")

    def search(
        self,
        query: str,
        top_k: int = 10,
        schedule_filter: Dict[str, Any] = None,
        preference_filter: Dict[str, float] = None,
    ) -> Dict[str, Any]:
        """Run retrieval, filtering, reranking, and final blended scoring."""
        collection_size = self.collection.count()
        initial_k = min(collection_size, max(top_k * 25, 120))
        results = self.collection.query(query_texts=[query], n_results=initial_k)

        diagnostics = {
            "retrieved": len(results["ids"][0]),
            "removed_by_schedule": 0,
            "removed_by_preferences": 0,
            "candidates_after_filters": 0,
            "returned": 0,
        }

        candidates = []
        for index, course_id in enumerate(results["ids"][0]):
            course = COURSES_BY_ID.get(course_id)
            if not course:
                continue

            schedule_info = analyze_schedule_fit(course, schedule_filter)
            if schedule_filter and not schedule_info["eligible"]:
                diagnostics["removed_by_schedule"] += 1
                continue

            if preference_filter and not matches_preferences(course, preference_filter):
                diagnostics["removed_by_preferences"] += 1
                continue

            candidates.append({
                "document": results["documents"][0][index],
                "course": course,
                "schedule_info": schedule_info,
                "keyword_score": score_keyword_overlap(query, course),
            })

        diagnostics["candidates_after_filters"] = len(candidates)
        if not candidates:
            return {"results": [], "diagnostics": diagnostics}

        pairs = [[query, candidate["document"]] for candidate in candidates]
        cross_scores = self.cross_encoder.predict(pairs)

        ranked_results = []
        for index, raw_score in enumerate(cross_scores):
            candidate = candidates[index]
            course = candidate["course"]
            schedule_info = candidate["schedule_info"]
            component_scores = blend_scores(
                semantic_score=normalize_cross_score(float(raw_score)),
                keyword_score=candidate["keyword_score"],
                schedule_score=schedule_info.get("score", 1.0),
                rating_score=score_rating(course),
                utility_score=score_utility(course),
                workload_score=score_workload(course, preference_filter),
                review_score=score_reviews(course),
            )
            ranked_results.append({
                "course": course,
                "raw_cross_score": float(raw_score),
                "display_score": int(round(component_scores["final"] * 100)),
                "scores": component_scores,
                "schedule_info": schedule_info,
                "reason_tags": build_reason_tags(course, component_scores, schedule_info),
            })

        ranked_results.sort(key=lambda item: item["scores"]["final"], reverse=True)
        diagnostics["returned"] = min(len(ranked_results), top_k)
        return {"results": ranked_results[:top_k], "diagnostics": diagnostics}

    def generate_json_advice(
        self,
        query: str,
        top_k: int = 5,
        schedule_filter: Dict[str, Any] = None,
        preference_filter: Dict[str, float] = None,
        search_payload: Dict[str, Any] = None,
    ) -> Dict[str, Any]:
        """Ask the LLM for a compact JSON summary grounded in retrieved results."""
        if not self.use_groq:
            return {"error": "Groq API not configured."}

        payload = search_payload or self.search(
            query,
            top_k=top_k,
            schedule_filter=schedule_filter,
            preference_filter=preference_filter,
        )
        hits = payload.get("results", [])
        if not hits:
            return {
                "recommended_courses": [],
                "explanation": "No matching courses found for the current goal and active filters.",
                "reason_per_course": [],
            }

        context_blocks = []
        for item in hits:
            course = item["course"]
            description = (course.get("description_clean") or course.get("description") or "")[:280]
            score_summary = item.get("scores", {})
            review_summary = (
                f"rating={course.get('avg_rating')}, workload={course.get('avg_workload_hours')}, "
                f"reviews={course.get('review_count', 0)}"
            )
            fit_summary = (
                f"match={item.get('display_score')} "
                f"topic={int(round(score_summary.get('semantic', 0) * 100))} "
                f"schedule={int(round(score_summary.get('schedule', 0) * 100))}"
            )
            context_blocks.append(
                f"ID: {course.get('course_id')} | Title: {course.get('course_name')} | "
                f"Level: {course.get('level')} | Time: {course.get('_meetingTime')} | "
                f"{review_summary} | {fit_summary} | Reasons: {', '.join(item.get('reason_tags', []))} | "
                f"Desc: {description}"
            )
        context_str = "\n".join(context_blocks)

        prompt = f"""You are an academic advisor. Based strictly on the provided context, recommend courses for the student.
Return valid JSON with exactly these keys:
- recommended_courses: a list of course IDs.
- explanation: a concise overall recommendation summary.
- reason_per_course: a list of objects with keys course_id, rationale, and confidence.

Use the provided fit summaries and do not invent attributes that are not present in the context.
If the context is insufficient, return an empty recommended_courses list instead of inventing courses.

Query: {query}
Context:
{context_str}
"""
        response = self.llm_client.chat.completions.create(
            model=self.llm_model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.2,
            max_tokens=700,
        )
        return json.loads(response.choices[0].message.content)


In [6]:
# Initialize the retrieval system and populate the vector store when needed.
rag_system = CourseRAG(db_path=str(DB_PATH), use_groq=USE_GROQ)
rag_system.ingest_data(COURSES)


Loading Bi-Encoder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Cross-Encoder for re-ranking...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection already populated with 3145 items. Skipping ingest.


## 5. UI Controls and Styling
Create the search widgets and the visual shell for the interactive course exploration experience.


In [7]:
# Create the main query input and filter widgets.
query_input = widgets.Text(
    value='I want to learn about product management and prototyping',
    placeholder='Describe what you want to learn, build, or explore...',
    description='',
    layout=widgets.Layout(width='100%'),
)
query_input.style = {'description_width': '0px'}
query_input.add_class('course-search-input')

day_selector = widgets.SelectMultiple(
    options=[('Monday', 'M'), ('Tuesday', 'T'), ('Wednesday', 'W'), ('Thursday', 'R'), ('Friday', 'F')],
    value=(),
    description='Days:',
    layout=widgets.Layout(width='260px', height='150px'),
)

time_window_input = widgets.Text(
    value='',
    placeholder='e.g. 9:00 AM-12:00 PM',
    description='Free Time:',
    layout=widgets.Layout(width='260px'),
)

top_k_slider = widgets.IntSlider(
    value=5,
    min=3,
    max=10,
    step=1,
    description='Results:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px'),
)

min_rating_slider = widgets.FloatSlider(
    value=0,
    min=0,
    max=5,
    step=0.5,
    description='Min Rating:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px'),
)

max_workload_slider = widgets.FloatSlider(
    value=20,
    min=2,
    max=25,
    step=1,
    description='Max Workload:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px'),
)

min_reviews_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=10,
    step=1,
    description='Min Reviews:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px'),
)

search_btn = widgets.Button(
    description='Search & Analyze',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='220px', height='54px'),
)
search_btn.style.button_color = '#173f35'
search_btn.add_class('course-search-button')

status_html = widgets.HTML()
output_area = widgets.HTML()


In [8]:
# Centralize the dashboard styling so UI updates stay easy to manage.
hero_css = widgets.HTML(
    '''
<style>
.course-search-hero {
  background: linear-gradient(135deg, #fff7ec 0%, #f4eadc 52%, #efe5d7 100%);
  border: 1px solid #e6d7c2;
  border-radius: 30px;
  padding: 28px 28px 24px 28px;
  box-shadow: 0 20px 44px rgba(69, 46, 20, 0.08);
  margin-bottom: 16px;
}
.course-search-hero h2 {
  margin: 0;
  font-size: 30px;
  color: #2f2418;
  letter-spacing: -0.02em;
}
.course-search-hero p {
  margin: 8px 0 0 0;
  color: #675847;
  line-height: 1.7;
  max-width: 760px;
}
.course-search-kicker {
  font-size: 12px;
  text-transform: uppercase;
  letter-spacing: 0.14em;
  color: #8f6b45;
  margin-bottom: 10px;
}
.course-search-shell {
  background: rgba(255, 253, 248, 0.92);
  border: 1px solid #eadfce;
  border-radius: 24px;
  padding: 16px;
  box-shadow: inset 0 1px 0 rgba(255,255,255,0.55);
}
.course-search-label {
  font-size: 12px;
  text-transform: uppercase;
  letter-spacing: 0.12em;
  color: #8d6e4a;
  margin-bottom: 10px;
}
.course-search-input input {
  min-height: 56px !important;
  border-radius: 18px !important;
  border: 1px solid #d9c7ae !important;
  background: #fffdfa !important;
  box-shadow: inset 0 1px 2px rgba(73, 52, 24, 0.04);
  font-size: 15px !important;
  padding: 0 18px !important;
  color: #2f2418 !important;
}
.course-search-input input:focus {
  border-color: #175c4d !important;
  box-shadow: 0 0 0 4px rgba(23, 92, 77, 0.12) !important;
}
.course-search-button button {
  border-radius: 18px !important;
  border: none !important;
  font-weight: 700 !important;
  letter-spacing: 0.01em;
  box-shadow: 0 12px 24px rgba(23, 63, 53, 0.18);
}
</style>
'''
)

hero_intro = widgets.HTML(
    '''
<section class="course-search-hero">
  <div class="course-search-kicker">Course Discovery Studio</div>
  <h2>Find Your Courses</h2>
  <p>Turn a broad learning goal into a shortlist of realistic course options, then refine the list by schedule, workload, rating quality, and review signal.</p>
</section>
'''
)


## 6. UI Helper Functions and Rendering
Separate filtering, formatting, and HTML rendering logic so the event handler stays short and readable.


In [9]:
def build_schedule_filter(selected_days, time_window: str):
    """Convert widget values into the schedule filter expected by retrieval."""
    clean_window = (time_window or '').strip()
    time_slots = []

    if clean_window:
        if '-' not in clean_window:
            raise ValueError('Time window should look like 9:00 AM-12:00 PM')
        start_str, end_str = [part.strip() for part in clean_window.split('-', 1)]
        time_slots = parse_time_slots(start_str, end_str)
        if not time_slots:
            raise ValueError('Could not parse the selected time window.')

    if not selected_days and not time_slots:
        return None

    return {
        'selected_days': list(selected_days),
        'time_slots': time_slots,
    }


def build_preference_filter() -> Dict[str, float]:
    """Read slider values into a simple preference dictionary."""
    return {
        'min_rating': float(min_rating_slider.value),
        'max_workload': float(max_workload_slider.value),
        'min_reviews': int(min_reviews_slider.value),
    }


def format_metric(value, suffix: str = '') -> str:
    """Format scalar values for display cards and tables."""
    if value is None or value == '':
        return 'N/A'
    if isinstance(value, float):
        value = round(value, 2)
    return f"{value}{suffix}"


def build_badge(text, tone: str = 'neutral') -> str:
    """Render a small colored badge for metadata and explanations."""
    palette = {
        'neutral': ('#f4efe6', '#5f4b32'),
        'accent': ('#dcefe8', '#175c4d'),
        'warm': ('#f8dfd2', '#8f3c22'),
        'strong': ('#173f35', '#f5efe3'),
    }
    background, foreground = palette.get(tone, palette['neutral'])
    return (
        f"<span style='display:inline-block;padding:6px 10px;border-radius:999px;"
        f"background:{background};color:{foreground};font-size:12px;font-weight:600;"
        f"margin:4px 6px 0 0;'>{escape(str(text))}</span>"
    )


def build_progress_bar(value: float, color: str) -> str:
    """Visualize normalized scores with a compact horizontal bar."""
    width = max(0, min(int(round(value * 100)), 100))
    return (
        f"<div style='height:8px;background:#eadfce;border-radius:999px;overflow:hidden;'>"
        f"<div style='height:100%;width:{width}%;background:{color};border-radius:999px;'></div></div>"
    )


def build_match_label(score: int) -> str:
    if score >= 85:
        return 'Excellent fit'
    if score >= 70:
        return 'Strong fit'
    if score >= 55:
        return 'Possible fit'
    return 'Exploratory fit'


In [10]:
def render_ai_advice(advice_json: Dict[str, Any]) -> str:
    """Show the optional LLM-generated summary in a highlighted block."""
    if not advice_json or advice_json.get('error'):
        return ''

    explanation = escape(str(advice_json.get('explanation', '')))
    reasons = advice_json.get('reason_per_course', []) or []
    reason_items = ''.join(
        f"<li><strong>{escape(str(item.get('course_id', 'Unknown')))}</strong>: "
        f"{escape(str(item.get('rationale', '')))} "
        f"<span style='color:#d4c9b6'>(confidence: {escape(str(item.get('confidence', 'n/a')))} )</span></li>"
        for item in reasons
    )

    if not explanation and not reason_items:
        return ''

    return f"""
    <section style='background:linear-gradient(135deg,#173f35,#245446);color:#f7f1e8;border-radius:22px;padding:22px 24px;margin-bottom:18px;box-shadow:0 16px 36px rgba(19,63,53,0.16);'>
      <div style='font-size:12px;letter-spacing:0.12em;text-transform:uppercase;opacity:0.78;margin-bottom:8px;'>AI Recommendation Summary</div>
      <div style='font-size:16px;line-height:1.7;margin-bottom:12px;'>{explanation}</div>
      {f'<ul style="margin:0;padding-left:18px;line-height:1.7;">{reason_items}</ul>' if reason_items else ''}
    </section>
    """


def render_comparison_table(results: List[Dict[str, Any]]) -> str:
    """Compare the top few recommendations in a compact table."""
    top_results = results[:3]
    if not top_results:
        return ''

    rows = []
    for item in top_results:
        course = item['course']
        rows.append(f"""
        <tr>
          <td style='padding:10px 12px;border-top:1px solid #eadfce;color:#2f2418;font-weight:600;'>{escape(str(course.get('course_id', 'N/A')))}</td>
          <td style='padding:10px 12px;border-top:1px solid #eadfce;color:#5d5143;'>{escape(str(course.get('course_name', 'Unknown')))}</td>
          <td style='padding:10px 12px;border-top:1px solid #eadfce;color:#175c4d;font-weight:700;'>{item.get('display_score', 0)}</td>
          <td style='padding:10px 12px;border-top:1px solid #eadfce;color:#5d5143;'>{format_metric(course.get('avg_rating'), '/5')}</td>
          <td style='padding:10px 12px;border-top:1px solid #eadfce;color:#5d5143;'>{format_metric(course.get('avg_workload_hours'), ' hrs')}</td>
          <td style='padding:10px 12px;border-top:1px solid #eadfce;color:#5d5143;'>{escape(str(course.get('_meetingTime', 'TBA')))}</td>
        </tr>
        """)

    return f"""
    <section style='background:#fffdf8;border:1px solid #eadfce;border-radius:24px;padding:18px 20px;margin-bottom:18px;box-shadow:0 12px 30px rgba(73,52,24,0.06);'>
      <div style='font-size:12px;letter-spacing:0.12em;text-transform:uppercase;color:#8f6b45;margin-bottom:10px;'>Quick Compare</div>
      <table style='width:100%;border-collapse:collapse;font-size:14px;'>
        <thead>
          <tr style='text-align:left;color:#7a6f61;'>
            <th style='padding:0 12px 10px 12px;'>Course</th>
            <th style='padding:0 12px 10px 12px;'>Title</th>
            <th style='padding:0 12px 10px 12px;'>Match</th>
            <th style='padding:0 12px 10px 12px;'>Rating</th>
            <th style='padding:0 12px 10px 12px;'>Workload</th>
            <th style='padding:0 12px 10px 12px;'>Time</th>
          </tr>
        </thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </section>
    """


In [11]:
def render_course_cards(
    search_payload: Dict[str, Any],
    query: str,
    schedule_filter: Dict[str, Any] = None,
    preference_filter: Dict[str, float] = None,
    advice_json: Dict[str, Any] = None,
) -> str:
    """Render the full search results dashboard as HTML."""
    results = search_payload.get('results', [])
    diagnostics = search_payload.get('diagnostics', {})
    cards = []

    for rank, item in enumerate(results, start=1):
        course = item['course']
        scores = item.get('scores', {})
        course_id = escape(str(course.get('course_id', 'N/A')))
        name = escape(str(course.get('course_name', 'Unknown')))
        description = escape(str((course.get('description_clean') or course.get('description') or 'No description available.'))[:320])
        level = build_badge(course.get('level', 'Unknown level'), 'accent')
        industry = build_badge(course.get('industry', 'Unknown domain'), 'neutral')
        meeting = build_badge(course.get('_meetingTime', 'TBA'), 'warm')
        match_label = build_badge(build_match_label(item.get('display_score', 0)), 'strong')
        skills = ''.join(build_badge(skill, 'neutral') for skill in (course.get('skills') or [])[:5])
        review = escape(' | '.join(course.get('review_snippets', [])[:2]))
        prerequisites = escape(str(course.get('prerequisites') or 'None listed'))
        reason_tags = ''.join(build_badge(reason, 'accent') for reason in item.get('reason_tags', []))
        schedule_summary = escape(str(item.get('schedule_info', {}).get('summary', 'No schedule filter applied.')))

        cards.append(f"""
        <article style='background:#fffdf8;border:1px solid #eadfce;border-radius:24px;padding:22px 22px 20px 22px;box-shadow:0 12px 30px rgba(73,52,24,0.08);margin-bottom:16px;'>
          <div style='display:flex;justify-content:space-between;gap:16px;align-items:flex-start;flex-wrap:wrap;'>
            <div>
              <div style='font-size:12px;letter-spacing:0.12em;text-transform:uppercase;color:#8d6e4a;margin-bottom:8px;'>Recommendation #{rank}</div>
              <h3 style='margin:0 0 8px 0;font-size:24px;color:#2f2418;'>{course_id} - {name}</h3>
              <div style='margin-bottom:6px;'>{match_label}</div>
            </div>
            <div style='background:#f5ede1;border-radius:18px;padding:10px 14px;text-align:right;min-width:140px;'>
              <div style='font-size:12px;color:#7f6b56;'>Weighted match</div>
              <div style='font-size:28px;font-weight:700;color:#1b5e52;'>{item.get('display_score', 0)}</div>
              <div style='font-size:12px;color:#7f6b56;'>out of 100</div>
            </div>
          </div>
          <div style='margin:8px 0 10px 0;'>{level}{industry}{meeting}</div>
          <div style='margin:0 0 12px 0;'>{reason_tags}</div>
          <p style='margin:8px 0 14px 0;color:#55483a;line-height:1.7;'>{description}</p>
          <div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin:12px 0 14px 0;'>
            <div style='background:#f8f4ec;border-radius:16px;padding:12px 14px;'><div style='font-size:12px;color:#7a6f61;margin-bottom:6px;'>Topic fit</div><div style='font-size:20px;font-weight:700;color:#2f2418;margin-bottom:8px;'>{int(round(scores.get('semantic', 0) * 100))}</div>{build_progress_bar(scores.get('semantic', 0), '#175c4d')}</div>
            <div style='background:#f8f4ec;border-radius:16px;padding:12px 14px;'><div style='font-size:12px;color:#7a6f61;margin-bottom:6px;'>Keyword overlap</div><div style='font-size:20px;font-weight:700;color:#2f2418;margin-bottom:8px;'>{int(round(scores.get('keyword', 0) * 100))}</div>{build_progress_bar(scores.get('keyword', 0), '#8f6b45')}</div>
            <div style='background:#f8f4ec;border-radius:16px;padding:12px 14px;'><div style='font-size:12px;color:#7a6f61;margin-bottom:6px;'>Schedule fit</div><div style='font-size:20px;font-weight:700;color:#2f2418;margin-bottom:8px;'>{int(round(scores.get('schedule', 0) * 100))}</div>{build_progress_bar(scores.get('schedule', 0), '#b86b3f')}</div>
            <div style='background:#f8f4ec;border-radius:16px;padding:12px 14px;'><div style='font-size:12px;color:#7a6f61;'>Student signal</div><div style='font-size:20px;font-weight:700;color:#2f2418;'>{int(round(((scores.get('rating', 0) + scores.get('utility', 0) + scores.get('reviews', 0)) / 3) * 100))}</div><div style='font-size:12px;color:#7a6f61;margin-top:4px;'>rating, utility, review volume</div></div>
          </div>
          <div style='background:#fbf7ef;border:1px solid #eadfce;border-radius:18px;padding:14px 16px;margin-bottom:14px;'>
            <div style='font-size:12px;letter-spacing:0.08em;text-transform:uppercase;color:#8f6b45;margin-bottom:8px;'>Why this matched</div>
            <div style='color:#5b4a38;line-height:1.7;'>
              <strong>Schedule:</strong> {schedule_summary}<br>
              <strong>Rating:</strong> {format_metric(course.get('avg_rating'), '/5')} | <strong>Workload:</strong> {format_metric(course.get('avg_workload_hours'), ' hrs/wk')} | <strong>Reviews:</strong> {format_metric(course.get('review_count'))}
            </div>
          </div>
          <div style='margin:0 0 12px 0;'><strong style='color:#5b4a38;'>Skills:</strong> {skills or '<span style="color:#8a7b68;">No structured skills listed.</span>'}</div>
          <div style='margin:0 0 8px 0;color:#5b4a38;'><strong>Prerequisites:</strong> {prerequisites}</div>
          <div style='margin:0;color:#5b4a38;'><strong>Student voice:</strong> {review or 'No review snippet available.'}</div>
        </article>
        """)

    selected_days = ', '.join(schedule_filter.get('selected_days', [])) if schedule_filter and schedule_filter.get('selected_days') else 'Any day'
    schedule_text = escape(time_window_input.value.strip() or 'Any time')
    query_text = escape(query)
    summary_badges = ''.join([
        build_badge(f"{len(results)} results", 'accent'),
        build_badge(f"Retrieved {diagnostics.get('retrieved', 0)} candidates", 'neutral'),
        build_badge(f"Min rating {preference_filter.get('min_rating', 0)}", 'neutral'),
        build_badge(f"Max workload {preference_filter.get('max_workload', 'Any')} hrs", 'neutral'),
        build_badge(f"Min reviews {preference_filter.get('min_reviews', 0)}", 'warm'),
        build_badge(f"Days: {selected_days}", 'neutral'),
        build_badge(f"Window: {schedule_text}", 'neutral'),
    ])

    funnel_text = (
        f"Started with {diagnostics.get('retrieved', 0)} semantic matches, removed "
        f"{diagnostics.get('removed_by_schedule', 0)} on schedule and "
        f"{diagnostics.get('removed_by_preferences', 0)} on preference thresholds, "
        f"then reranked {diagnostics.get('candidates_after_filters', 0)} viable courses."
    )

    return f"""
    <div style='font-family:Helvetica,Arial,sans-serif;background:linear-gradient(180deg,#f6f1e8 0%,#fbfaf6 100%);padding:22px;border-radius:28px;border:1px solid #eadfce;'>
      <section style='background:linear-gradient(135deg,#fff8ee,#f3eadf);border:1px solid #eadfce;border-radius:24px;padding:22px 24px;margin-bottom:18px;'>
        <div style='font-size:12px;letter-spacing:0.14em;text-transform:uppercase;color:#8f6b45;margin-bottom:10px;'>Course Match Dashboard</div>
        <h2 style='margin:0 0 8px 0;font-size:28px;color:#2f2418;'>Results for: {query_text}</h2>
        <p style='margin:0;color:#5d5143;line-height:1.7;'>This ranking blends semantic retrieval, keyword overlap, schedule fit, and student signals, so the order reflects both relevance and practicality.</p>
        <div style='margin-top:12px;'>{summary_badges}</div>
        <div style='margin-top:14px;background:#fffdf8;border-radius:18px;padding:14px 16px;color:#5b4a38;line-height:1.7;border:1px solid #eadfce;'><strong>Ranking funnel:</strong> {escape(funnel_text)}</div>
      </section>
      {render_ai_advice(advice_json)}
      {render_comparison_table(results)}
      {''.join(cards)}
    </div>
    """


## 7. Search Interaction and Display
Keep the event handler focused on orchestration: validate inputs, search, optionally call the LLM, and render results.


In [12]:
def render_message_card(title: str, body: str, tone: str = 'neutral') -> str:
    """Render status and error messages in a consistent card style."""
    palettes = {
        'neutral': ('#fff8ee', '#2f2418', '#eadfce'),
        'error': ('#fdecea', '#7b241c', '#f3c4bf'),
    }
    background, foreground, border = palettes[tone]
    return (
        f"<div style='font-family:Helvetica,Arial,sans-serif;background:{background};color:{foreground};"
        f"padding:18px 20px;border-radius:22px;border:1px solid {border};'>"
        f"<strong style='display:block;margin-bottom:6px;'>{escape(title)}</strong>"
        f"<span>{escape(body)}</span></div>"
    )


def build_no_match_message(diagnostics: Dict[str, Any]) -> str:
    """Explain why no courses survived the active filters."""
    return (
        f"No courses survived the current filters. The system retrieved {diagnostics.get('retrieved', 0)} relevant candidates, "
        f"but removed {diagnostics.get('removed_by_schedule', 0)} for schedule mismatch and "
        f"{diagnostics.get('removed_by_preferences', 0)} for rating/workload/review thresholds. "
        "Try widening the time window, selecting fewer days, or lowering one of the preference thresholds."
    )


def on_search_clicked(_button) -> None:
    """Handle the full search flow from widget input to rendered results."""
    status_html.value = render_message_card(
        'Working',
        'Searching, scoring with blended signals, and formatting the strongest course matches...',
    )
    output_area.value = ''

    query = query_input.value.strip()
    if not query:
        status_html.value = render_message_card('Missing Goal', 'Please enter a learning goal before searching.', 'error')
        return

    try:
        schedule_filter = build_schedule_filter(day_selector.value, time_window_input.value)
    except ValueError as exc:
        status_html.value = render_message_card('Schedule Filter Error', str(exc), 'error')
        return

    preference_filter = build_preference_filter()
    search_payload = rag_system.search(
        query,
        top_k=int(top_k_slider.value),
        schedule_filter=schedule_filter,
        preference_filter=preference_filter,
    )
    results = search_payload.get('results', [])

    if not results:
        status_html.value = render_message_card(
            'No Matches',
            build_no_match_message(search_payload.get('diagnostics', {})),
            'error',
        )
        return

    advice_json = None
    if USE_GROQ:
        try:
            advice_json = rag_system.generate_json_advice(
                query,
                top_k=int(top_k_slider.value),
                schedule_filter=schedule_filter,
                preference_filter=preference_filter,
                search_payload=search_payload,
            )
        except Exception as exc:
            advice_json = {
                'explanation': f'AI summary unavailable: {exc}',
                'reason_per_course': [],
            }

    status_html.value = ''
    output_area.value = render_course_cards(
        search_payload,
        query,
        schedule_filter=schedule_filter,
        preference_filter=preference_filter,
        advice_json=advice_json,
    )


search_btn.on_click(on_search_clicked)


In [13]:
# Assemble and display the dashboard layout.
controls_col_1 = widgets.VBox([day_selector, time_window_input], layout=widgets.Layout(gap='10px'))
controls_col_2 = widgets.VBox([top_k_slider, min_rating_slider, max_workload_slider, min_reviews_slider], layout=widgets.Layout(gap='10px'))

search_row = widgets.HBox(
    [query_input, search_btn],
    layout=widgets.Layout(gap='14px', align_items='center', flex_flow='row wrap', width='100%'),
)

search_card = widgets.VBox(
    [
        widgets.HTML(
            '<div class="course-search-label">Learning Goal</div>'
            '<div style="color:#6d5d4d;line-height:1.6;">'
            'Type a goal like product management, NLP, human-centered design, or startup prototyping.'
            '</div>'
        ),
        search_row,
    ],
    layout=widgets.Layout(gap='12px'),
)
search_card.add_class('course-search-shell')

display(
    widgets.VBox(
        [
            hero_css,
            hero_intro,
            search_card,
            widgets.HBox([controls_col_1, controls_col_2], layout=widgets.Layout(gap='18px', flex_flow='row wrap')),
            status_html,
            output_area,
        ],
        layout=widgets.Layout(gap='16px'),
    )
)


## 8. Evaluation

We evaluate CoursePilot on five hand-curated queries that span different
domains. For each query we collect three deterministic metrics computed in
Python — department hit rate, top-3 coverage, and citation validity — plus one
LLM-judged metric (1–5 query–recommendation relevance) using Llama 3.3 as the
judge. Results are aggregated into per-query and system-wide tables.

In [14]:
import re
import time

In [15]:
# Test queries with expected department prefixes

TEST_QUERIES = [
    {
        "query": "machine learning for natural language processing",
        "expected_depts": ["10", "11"],
        "note": "Core ML and LTI courses",
    },
    {
        "query": "product management and prototyping for tech",
        "expected_depts": ["05", "17", "70", "94"],
        "note": "Cross-disciplinary, HCI + Heinz + Tepper",
    },
    {
        "query": "data visualization for public policy",
        "expected_depts": ["90", "94", "16"],
        "note": "Non-STEM, Heinz / Information Systems",
    },
    {
        "query": "introduction to artificial intelligence ethics",
        "expected_depts": ["80", "19", "76", "17"],
        "note": "Philosophy / EPP / English / SE",
    },
    {
        "query": "feedback control systems and robotics",
        "expected_depts": ["24", "16", "18"],
        "note": "MechE + Robotics + ECE",
    },
]


def get_dept_prefix(course_id: str) -> str:
    """Extract the 2-digit department prefix from a course ID like '15-213'."""
    match = re.match(r"\s*(\d{2})", str(course_id or ""))
    return match.group(1) if match else ""

In [18]:
# LLM-as-judge: returns an integer score from 1 to 5

def llm_judge_relevance(query: str, course_title: str,
                         course_desc: str, rationale: str) -> int:
    """Ask Llama 3.3 to score query-recommendation relevance on a 1-5 scale."""
    prompt = (
        f"You are evaluating an academic course recommendation system.\n\n"
        f"User query: {query}\n"
        f"Recommended course: {course_title}\n"
        f"Description: {course_desc[:500]}\n"
        f"System reason: {rationale}\n\n"
        f"On a scale of 1 to 5, how well does this recommendation match the user query?\n"
        f"1 = irrelevant, 3 = somewhat relevant, 5 = excellent match.\n"
        f"Reply with ONLY a single integer between 1 and 5. No other text."
    )
    try:
        response = rag_system.llm_client.chat.completions.create(
            model=rag_system.llm_model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=5,
        )
        text = response.choices[0].message.content.strip()
        digit = re.search(r"[1-5]", text)
        return int(digit.group()) if digit else None
    except Exception as exc:
        print(f"  Judge call failed: {exc}")
        return None

In [19]:
# Run all queries and collect per-recommendation rows

print("Running evaluation on 5 queries...\n")
eval_rows = []
per_query_summary = []

for tc_idx, tc in enumerate(TEST_QUERIES, start=1):
    query = tc["query"]
    expected = set(tc["expected_depts"])
    print(f"[{tc_idx}/5] {query}")

    # Run retrieval
    payload = rag_system.search(query, top_k=5)
    hits = payload.get("results", [])

    # Get LLM-cited recommendations (so we can check citation validity)
    advice = rag_system.generate_json_advice(query, top_k=5, search_payload=payload)
    cited_ids = set(advice.get("recommended_courses", []))
    rationales = {r["course_id"]: r.get("rationale", "")
                  for r in advice.get("reason_per_course", [])}

    retrieved_ids = {item["course"].get("course_id", "") for item in hits}
    cite_valid = (1 if cited_ids and cited_ids.issubset(retrieved_ids)
                  else 0 if cited_ids else None)

    # Track top-3 hit at the query level
    top3_depts = [get_dept_prefix(item["course"].get("course_id", ""))
                  for item in hits[:3]]
    top3_hit = int(any(d in expected for d in top3_depts))

    # Per-recommendation metrics (top-5)
    for rank, item in enumerate(hits[:5], start=1):
        course = item["course"]
        cid = str(course.get("course_id", ""))
        dept = get_dept_prefix(cid)
        rationale = rationales.get(cid, "(not cited by LLM)")
        desc = course.get("description_clean") or course.get("description") or ""

        judge_score = llm_judge_relevance(
            query=query,
            course_title=course.get("course_name", ""),
            course_desc=desc,
            rationale=rationale,
        )

        eval_rows.append({
            "query": query,
            "rank": rank,
            "course_id": cid,
            "title": course.get("course_name", "")[:50],
            "dept": dept,
            "dept_hit": int(dept in expected),
            "match_score": item.get("display_score", 0),
            "judge_relevance": judge_score,
            "cited_by_llm": int(cid in cited_ids),
        })
        time.sleep(0.3)  # be polite to the API

    per_query_summary.append({
        "query": query,
        "top3_hit": top3_hit,
        "cite_valid": cite_valid,
        "n_retrieved": payload["diagnostics"]["retrieved"],
        "n_after_filters": payload["diagnostics"]["candidates_after_filters"],
    })
    print(f"  Top-3 hit: {bool(top3_hit)} | Citation valid: {cite_valid}")

print("\nEvaluation complete.\n")

Running evaluation on 5 queries...

[1/5] machine learning for natural language processing
  Top-3 hit: True | Citation valid: 1
[2/5] product management and prototyping for tech
  Top-3 hit: False | Citation valid: 1
[3/5] data visualization for public policy
  Top-3 hit: True | Citation valid: 1
[4/5] introduction to artificial intelligence ethics
  Top-3 hit: False | Citation valid: 1
[5/5] feedback control systems and robotics
  Top-3 hit: True | Citation valid: 1

Evaluation complete.



In [20]:
# Aggregate into tables

df_rows = pd.DataFrame(eval_rows)
df_summary = pd.DataFrame(per_query_summary)

# Per-query mean of recommendation-level metrics
per_query_means = (
    df_rows.groupby("query")
           .agg(
               mean_dept_hit=("dept_hit", "mean"),
               mean_match_score=("match_score", "mean"),
               mean_judge_relevance=("judge_relevance", "mean"),
           )
           .round(2)
           .reset_index()
)
per_query_table = df_summary.merge(per_query_means, on="query")

print("=" * 80)
print("PER-QUERY RESULTS")
print("=" * 80)
print(per_query_table.to_string(index=False))

# System-wide numbers
print("\n" + "=" * 80)
print("SYSTEM-WIDE METRICS")
print("=" * 80)
print(f"Department hit rate (top-5 mean):       "
      f"{df_rows['dept_hit'].mean():.2f}")
print(f"Top-3 coverage rate (any expected dept): "
      f"{df_summary['top3_hit'].mean():.2f}")
print(f"Citation validity rate:                  "
      f"{df_summary['cite_valid'].dropna().mean():.2f}")
print(f"LLM-judged relevance (mean of 1-5):      "
      f"{df_rows['judge_relevance'].dropna().mean():.2f}")
print(f"Total recommendations evaluated:         {len(df_rows)}")

# Save tables for the report
df_rows.to_csv("eval_per_recommendation.csv", index=False)
per_query_table.to_csv("eval_per_query.csv", index=False)
print("\nSaved eval_per_recommendation.csv and eval_per_query.csv")

PER-QUERY RESULTS
                                           query  top3_hit  cite_valid  n_retrieved  n_after_filters  mean_dept_hit  mean_match_score  mean_judge_relevance
machine learning for natural language processing         1           1          125              125            0.8              86.8                   3.8
     product management and prototyping for tech         0           1          125              125            0.0              80.0                   3.6
            data visualization for public policy         1           1          125              125            0.6              84.0                   3.8
  introduction to artificial intelligence ethics         0           1          125              125            0.0              87.2                   3.2
           feedback control systems and robotics         1           1          125              125            1.0              82.8                   4.6

SYSTEM-WIDE METRICS
Department hit rate (top-